# Scikit-learn

Advanced reference for preprocessing, pipelines, model selection, evaluation, feature selection, and unsupervised learning.

## Split data correctly

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)

`train_test_split()` separates observations into subsets. `stratify=y` preserves class proportions. Keep the test set untouched until final evaluation. For grouped or temporal observations, use group-aware or time-aware splitting rather than random splitting.

## Preprocessing pipeline

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

prep_model = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()), ('model', LogisticRegression(max_iter=1000))])

`Pipeline` chains transformations and estimators so each step is fitted within the training fold. `SimpleImputer` handles missing values; `StandardScaler` standardizes features. Other transformers include `MinMaxScaler`, `RobustScaler`, `PowerTransformer`, `OneHotEncoder`, and `OrdinalEncoder`.

## Column-specific transformations

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

transform = ColumnTransformer([('num', StandardScaler(), [0,1]), ('cat', OneHotEncoder(handle_unknown='ignore'), [2])])

`ColumnTransformer` applies separate transformations to selected columns. `OneHotEncoder(handle_unknown='ignore')` handles categories not seen during fitting. Use `remainder='passthrough'` to retain unlisted columns. Put this transformer inside a pipeline to avoid leakage.

## Fit and evaluate

In [ ]:
model = Pipeline([('scale', StandardScaler()), ('classifier', LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
model.predict_proba(X_test)[:3]

`fit()` learns parameters from training data. `predict()` returns class labels; `predict_proba()` returns probabilities when supported. The pipeline ensures preprocessing is applied consistently to new data.

## Classification metrics

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

accuracy_score(y_test, pred)
confusion_matrix(y_test, pred)
print(classification_report(y_test, pred))

`accuracy_score()` measures the fraction correct. `confusion_matrix()` shows true versus predicted classes. `classification_report()` includes precision, recall, F1, and support. For imbalanced data, consider macro F1, balanced accuracy, PR-AUC, and per-class metrics. ROC-AUC evaluates ranking, not a fixed threshold.

## Cross-validation strategies

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_validate(model, X, y, cv=cv, scoring=['accuracy','f1_macro'], return_train_score=True)

`StratifiedKFold` preserves class proportions. Other splitters include `KFold`, `GroupKFold`, `StratifiedGroupKFold`, `TimeSeriesSplit`, and `RepeatedStratifiedKFold`. Choose the splitter based on the data-generating process. Grouped subjects or repeated measurements should not leak across folds.

## Hyperparameter optimization

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

search = RandomizedSearchCV(model, {'classifier__C':[.01,.1,1,10]}, n_iter=4, cv=cv, scoring='f1_macro', n_jobs=-1, random_state=42).fit(X_train, y_train)
search.best_params_, search.best_score_

`GridSearchCV` evaluates every parameter combination; `RandomizedSearchCV` samples a fixed number. Pipeline parameters use double underscores, such as `classifier__C`. Nested cross-validation is useful when estimating performance after tuning. Do not use the final test set to choose hyperparameters.

## Feature selection

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE

selector = SelectKBest(f_classif, k=3).fit(X_train, y_train)
selector.get_support(), selector.transform(X_train)

`SelectKBest` ranks features using a univariate score such as `f_classif`, `chi2`, or mutual information. `RFE` recursively removes features using an estimator's importance. Fit selectors inside cross-validation pipelines; selecting features before splitting causes leakage.

## Compare estimators

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC

models = {'logistic': LogisticRegression(max_iter=1000), 'forest': RandomForestClassifier(random_state=42), 'svm': SVC()}

Common classifiers include linear models, SVMs, decision trees, random forests, gradient boosting, and nearest neighbors. Scale features for distance- and margin-based estimators. Tree-based estimators generally do not require standardization. Compare models with identical splits and metrics.

## Interpretation and persistence

In [ ]:
from sklearn.inspection import permutation_importance
import joblib

fitted = model.fit(X_train, y_train)
importance = permutation_importance(fitted, X_test, y_test, n_repeats=20, random_state=42)
joblib.dump(fitted, 'model.joblib')

`permutation_importance()` measures score decrease after shuffling a feature. Correlated predictors can obscure individual importance. `joblib` persists fitted estimators; load only trusted files because pickle-based formats can execute code. Record package versions and preprocessing assumptions.

## Unsupervised methods

In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X)
labels = KMeans(n_clusters=3, n_init='auto', random_state=42).fit_predict(X_scaled)
coordinates = PCA(n_components=2).fit_transform(X_scaled)

`KMeans` partitions data into a chosen number of clusters and is sensitive to scaling and initialization. `DBSCAN` finds density-connected groups and can label noise. `PCA` projects data onto directions of maximum variance. Other methods include agglomerative clustering, Gaussian mixtures, NMF, and TruncatedSVD.